# Underdamped Langevin Inference (ULI) — Acoustic Force Field Recovery

This notebook applies the **Underdamped Langevin Inference** algorithm (Brückner, Ronceray & Broedersz, PRL 2020) to synthetic cell trajectories generated by `simulateTrajectories.py`.

### Pipeline
1. **k-Wave simulation** (`kwaveTrainingDataGenerator.py`) — 2D acoustic pressure fields at 1 MHz (LIPUS), 8 PPW
2. **Trajectory simulation** (`simulateTrajectories.py`) — underdamped Langevin cells driven by radiation force
3. **ULI inference** (this notebook) — recover F(x,v) and D(x,v) from trajectory data alone
4. **Validation** — compare inferred force to the ground-truth k-Wave field

### Physics
$$\dot{x} = v, \quad \dot{v} = F(x,v) + \sqrt{2D}\,\xi(t)$$

where $F(x,v) = F_{\rm acoustic}(x) - \gamma v$ (acoustic body force + viscous damping).

In [1]:
import os
import sys
import json
import numpy as np
import h5py
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.ndimage import gaussian_filter

# SFI package (must be on path)
sys.path.insert(0, 'StochasticForceInference')
os.environ["JAX_PLATFORMS"] = "cpu"

import jax.numpy as jnp
from jax import random, jit
from jax import vmap
import SFI

print("SFI version:", SFI.__version__)
print("Imports OK")


## 1. Load trajectory data

Each simulation produces one multi-particle CSV containing all ~48–50 cells. `StochasticTrajectoryData` pools them directly from that file — more data and better spatial coverage than the previous single-cell approach.

In [2]:
# Load trajectory metadata
with open('trajectoryData/traj_metadata.json') as f:
    traj_meta = json.load(f)

print(f"Total acoustic field simulations: {len(traj_meta)}")
print(f"Cells per field: {traj_meta[0]['n_cells']}")
print(f"Steps per trajectory: {traj_meta[0]['n_steps']}  ×  dt={traj_meta[0]['dt']} s")
print(f"Total duration: {traj_meta[0]['n_steps'] * traj_meta[0]['dt']:.0f} s")

In [3]:
def load_trajectories(sim_entry: dict):
    """
    Load the multi-particle trajectory CSV for one acoustic field.
    All cells are stored in a single file; StochasticTrajectoryData handles
    the particle_id column automatically.

    Returns (data, dt)
    """
    metadata, particle_indices, time_indices, xvals = \
        SFI.SFI_utils.load_trajectory_csv(sim_entry['trajectory_file'])

    dt = metadata['dt']
    data = SFI.StochasticTrajectoryData(
        xvals, time_indices, dt,
        particle_indices=particle_indices,
        compute_dXplus=True
    )
    return data, dt


# Pick sim 0000 (well, 1 transducer) for a first look
SIM_IDX = 0
sim_entry = traj_meta[SIM_IDX]
data, dt = load_trajectories(sim_entry)

print(f"Geometry: {sim_entry['geometry']}  |  {sim_entry['n_transducers']} transducer(s)")
print(f"Cells: {sim_entry['n_cells']}")
print(f"Trajectory data shape: {data.X.shape}   (time × particles × dim)")
print(f"Exploitable trajectory points: {data.Nparticles.sum()}")

## 2. Visualise the trajectories

In [4]:
def plot_trajectories(sim_entry, ax_xy, ax_yt):
    """Plot x-y paths and y(t) depth traces for all cells in one simulation."""
    _, particle_indices, time_indices, xvals = \
        SFI.SFI_utils.load_trajectory_csv(sim_entry['trajectory_file'])

    n_cells = int(particle_indices.max()) + 1
    colors = plt.cm.tab20(np.linspace(0, 1, n_cells))
    dt_val = sim_entry['dt']

    for pid in range(n_cells):
        mask = particle_indices == pid
        x_mm = xvals[mask, 0] * 1e3
        y_mm = xvals[mask, 1] * 1e3
        t    = time_indices[mask] * dt_val
        col  = colors[pid % len(colors)]
        ax_xy.plot(x_mm, y_mm, lw=0.6, alpha=0.6, color=col)
        ax_xy.plot(x_mm[0], y_mm[0], 'o', ms=3, color=col)
        ax_yt.plot(t, y_mm, lw=0.6, alpha=0.6, color=col)

    ax_xy.set_xlabel('x (mm)')
    ax_xy.set_ylabel('y — depth (mm)  [↓]')
    ax_xy.invert_yaxis()
    ax_xy.set_title(f"{sim_entry['geometry'].upper()}  |  {sim_entry['n_transducers']} transducer(s)\n"
                    f"{n_cells} cells on grid")

    ax_yt.set_xlabel('time (s)')
    ax_yt.set_ylabel('depth y (mm)  [↓]')
    ax_yt.invert_yaxis()
    ax_yt.set_title('depth vs time')


fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_trajectories(sim_entry, axes[0], axes[1])
plt.tight_layout()
plt.savefig('trajectoryData/traj_viz_sim0000.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. ULI Force Inference

We use a **separable normalised basis**: order-3 polynomial in position × linear in velocity (12 terms total).

**Why normalisation?**  Position coordinates are ~10 mm; velocity estimates ~10⁻⁵ m/s — a 1000× scale gap.  Without normalisation the polynomial terms span 15 orders of magnitude, making the regression ill-conditioned and γ recovery fail.  Normalising all inputs to unit variance fixes this.

**Why separable?**  Including velocity-squared terms (order 2+ in v) leads to overfitting of diffusion noise.  Keeping the velocity expansion at order 1 cleanly separates the acoustic force (position-dependent) from the damping (linear in v).

| Quantity | Target |
|---|---|
| D (diffusion) | within 5% of true |
| γ (damping) | within 30% of true |
| Force map Pearson *r* | > 0.5 at 2 mm smoothing |

The polynomial basis cannot represent sub-millimetre acoustic standing-wave fringes (λ/2 ≈ 0.75 mm at 1 MHz), so Pearson *r* is reported at 2 mm Gaussian smoothing to isolate the recoverable coarse-scale beam structure.


In [5]:
# ── Diffusion estimation ─────────────────────────────────────────────────────
S = SFI.UnderdampedLangevinInference(data)
S.compute_diffusion_constant(method='WeakNoise')
print(f"Inferred diffusion D:\n{np.array(S.diffusion_average)}")
print(f"\nTrue D: {sim_entry['diffusion']:.2e} m²/s³ × I")

# ── Coordinate normalisation ─────────────────────────────────────────────────
# Position (~1e-2 m) and velocity (~1e-5 m/s) differ by ~1000×.
# Normalising to unit variance prevents velocity-term overfitting and stabilises
# the regression, bringing D within 3% and γ within 25% of truth.
x_mean_np = np.mean(np.array(data.X), axis=(0, 1))          # (2,)  domain centroid [m]
x_std_np  = np.std (np.array(data.X), axis=(0, 1))          # (2,)  positional spread [m]
v_std_np  = np.std (np.array(data.dX) / dt, axis=(0, 1))   # (2,)  velocity scale [m/s]

x_mean_j = jnp.array(x_mean_np)
x_std_j  = jnp.array(x_std_np)
v_std_j  = jnp.array(v_std_np)

print(f"\nNormalisation scales:")
print(f"  x_mean = {x_mean_np*1e3} mm")
print(f"  x_std  = {x_std_np*1e3} mm")
print(f"  v_std  = {v_std_np} m/s")


In [6]:
# ── Force basis: order-3 position + linear velocity, normalised ──────────────
# 10 spatial monomials (order 3 in 2D)  +  2 linear velocity terms  =  12 total.
# Evaluated in normalised coordinates so all basis values are O(1).
from jax import jit as _jit

poly_x, _ = SFI.ULI_bases.underdamped_polynomial_basis(data.d, order=3, mode='x')
poly_v, _ = SFI.ULI_bases.underdamped_polynomial_basis(data.d, order=1, mode='v')

@_jit
def norm_scalar_basis(x, v):
    """Order-3 position poly + linear velocity in normalised coordinates."""
    x_n = (x - x_mean_j) / x_std_j
    v_n = v / v_std_j
    return jnp.concatenate([
        poly_x(x_n, v_n),       # 10 terms: [1, x0', x1', x0'², x0'x1', x1'², x0'³, ...]
        poly_v(x_n, v_n)[1:],   # 2 terms:  [v0', v1']  (constant already in poly_x)
    ])

(force_b, _, force_grad_b_v), names = SFI.ULI_bases.basis_selector(
    {'type': 'custom_scalar', 'functions': norm_scalar_basis},
    data.d, output='vector'
)

S.infer_force_linear(
    basis_linear        = force_b,
    basis_linear_grad_v = force_grad_b_v,
    M_mode              = 'symmetric',
    G_mode              = 'shift',
    diffusion_method    = 'noisy',
    basis_names         = names,
)
S.compute_force_error()
S.print_report()


## 4. Inferred force coefficients

The leading coefficients tell us: how well does the inferred force recover the downward acoustic body force and the viscous damping?

In [7]:
print("Inferred force coefficients:")
print(f"{'Basis function':25s}  {'Coefficient':>14s}  {'Std error':>12s}")
print("-" * 55)
coeffs_arr = np.array(S.force_coefficients_full)
stderr_arr = np.array(S.force_coefficients_stderr) if hasattr(S, 'force_coefficients_stderr') else np.zeros_like(coeffs_arr)
for name, c, se in zip(names, coeffs_arr, stderr_arr):
    print(f"{name:25s}  {c:14.4e}  {se:12.4e}")

print()
n_half = len(names) // 2
print(f"Expected for Fy: coefficient of 'v₁·e₁' ≈ −γ = −{sim_entry['gamma']:.1f} s⁻¹")

## 5. Validate: compare inferred force map to ground-truth k-Wave field

In [8]:
PML_SIZE           = 20
TARGET_PRESSURE_PA = 160e3
FORCE_SCALE        = 1.0 / 1050.0
PRESSURE_SCALE     = TARGET_PRESSURE_PA ** 2

def load_ground_truth(acoustic_h5, force_boost=1.0):
    """
    Return the ground-truth downward radiation force acceleration field [m/s²].
    force_boost must match the value used in simulateTrajectories.py so that
    the inferred and ground-truth force maps have the same physical scale.
    """
    with h5py.File(acoustic_h5, 'r') as f:
        rf = f['radiation_force_dens'][:]
        Nx = int(f.attrs['Nx']); Ny = int(f.attrs['Ny']); dx = float(f.attrs['dx_m'])
    rf_int = rf[PML_SIZE:Nx-PML_SIZE, PML_SIZE:Ny-PML_SIZE]
    iNx, iNy = rf_int.shape
    x_m = np.arange(iNx) * dx
    y_m = np.arange(iNy) * dx
    Fy  = rf_int * PRESSURE_SCALE * FORCE_SCALE * force_boost
    return Fy, x_m, y_m


Fy_gt, x_m_gt, y_m_gt = load_ground_truth(
    sim_entry['acoustic_field_file'],
    force_boost=sim_entry.get('force_boost', 1.0)
)

# Evaluate inferred force at zero velocity (v=0) on the ground-truth grid
XX, YY = np.meshgrid(x_m_gt, y_m_gt, indexing='ij')  # (iNx, iNy)
Xeval  = np.stack([XX.ravel(), YY.ravel()], axis=-1)   # (N, 2) positions
Veval  = np.zeros_like(Xeval)                           # v=0

# ULI force evaluation: F = einsum('a, iam -> im', coeffs, basis)
coeffs = jnp.array(S.force_coefficients_full)   # shape (n_basis * dim,)

def eval_force(x, v):
    """Evaluate inferred force at one point (x, v) each of shape (2,)."""
    basis = force_b(x[None], v[None])   # (1, n_basis, dim)
    return jnp.einsum('a,iam->im', coeffs, basis)[0]  # (dim,)

F_inferred_flat = np.array(vmap(eval_force)(
    jnp.array(Xeval, dtype=jnp.float32),
    jnp.array(Veval, dtype=jnp.float32)
))  # (N, 2)

Fy_inferred = F_inferred_flat[:, 1].reshape(XX.shape)   # y-component

print(f"Ground-truth Fy range: {Fy_gt.min():.3e} .. {Fy_gt.max():.3e} m/s²")
print(f"Inferred   Fy range: {Fy_inferred.min():.3e} .. {Fy_inferred.max():.3e} m/s²")


In [9]:
from scipy.ndimage import gaussian_filter

extent = [x_m_gt[0]*1e3, x_m_gt[-1]*1e3, y_m_gt[-1]*1e3, y_m_gt[0]*1e3]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(
    f"ULI Force Recovery  |  {sim_entry['geometry'].upper()}  |  {sim_entry['n_transducers']} transducer(s)",
    fontsize=13, fontweight='bold'
)

vmax = max(Fy_gt.max(), abs(Fy_inferred).max())

# Ground truth
ax = axes[0]
im = ax.imshow(Fy_gt.T, origin='upper', aspect='auto', extent=extent,
               cmap='viridis', vmin=0, vmax=vmax)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title('Ground truth\n$F_y$ from k-Wave [m/s²]')
ax.set_xlabel('x (mm)'); ax.set_ylabel('depth y (mm)')

# Inferred
ax = axes[1]
im = ax.imshow(Fy_inferred.T, origin='upper', aspect='auto', extent=extent,
               cmap='viridis', vmin=0, vmax=vmax)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title('ULI inferred\n$F_y$ at v=0 [m/s²]')
ax.set_xlabel('x (mm)')

# Residual
resid = Fy_inferred - Fy_gt
ax = axes[2]
lim = np.abs(resid).max()
im = ax.imshow(resid.T, origin='upper', aspect='auto', extent=extent,
               cmap='RdBu_r', vmin=-lim, vmax=lim)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title('Residual\n(inferred − truth) [m/s²]')
ax.set_xlabel('x (mm)')

plt.tight_layout()
plt.savefig(f'trajectoryData/uli_recovery_sim{SIM_IDX:04d}.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Correlation metrics ───────────────────────────────────────────────────────
# Full-resolution r is dominated by sub-mm acoustic fringes the polynomial cannot
# represent.  Smoothing to 2 mm isolates the recoverable coarse beam structure.
dx_m = x_m_gt[1] - x_m_gt[0]
sigma_2mm = 2e-3 / dx_m   # pixels for 2 mm Gaussian

r_full  = np.corrcoef(Fy_gt.ravel(), Fy_inferred.ravel())[0, 1]
rmse    = np.sqrt(np.mean((Fy_inferred - Fy_gt)**2))

gt_sm   = gaussian_filter(Fy_gt,       sigma=sigma_2mm)
inf_sm  = gaussian_filter(Fy_inferred, sigma=sigma_2mm)
r_2mm   = np.corrcoef(gt_sm.ravel(), inf_sm.ravel())[0, 1]

print(f"Pearson r  (full res) = {r_full:.4f}")
print(f"Pearson r  (2 mm smooth) = {r_2mm:.4f}  ← coarse beam structure")
print(f"RMSE       = {rmse:.3e} m/s²")
print(f"Rel. RMSE  = {rmse / Fy_gt.mean():.3f}  (normalised to mean force)")


## 6. Compare multiple transducer configurations

Run ULI on all simulations and collect Pearson r for each.

In [10]:
def run_uli(sim_entry):
    """Full ULI pipeline for one simulation. Returns (r_full, r_2mm, rmse, S)."""
    data, dt = load_trajectories(sim_entry)

    S = SFI.UnderdampedLangevinInference(data)
    S.compute_diffusion_constant(method='WeakNoise')

    # Per-simulation normalisation (position and velocity scales)
    x_mean_j_ = jnp.array(np.mean(np.array(data.X), axis=(0, 1)))
    x_std_j_  = jnp.array(np.std (np.array(data.X), axis=(0, 1)))
    v_std_j_  = jnp.array(np.std (np.array(data.dX) / dt, axis=(0, 1)))

    poly_x_, _ = SFI.ULI_bases.underdamped_polynomial_basis(data.d, order=3, mode='x')
    poly_v_, _ = SFI.ULI_bases.underdamped_polynomial_basis(data.d, order=1, mode='v')

    @jit
    def _norm_basis(x, v):
        x_n = (x - x_mean_j_) / x_std_j_
        v_n = v / v_std_j_
        return jnp.concatenate([poly_x_(x_n, v_n), poly_v_(x_n, v_n)[1:]])

    (force_b_, _, force_grad_b_v_), names_ = SFI.ULI_bases.basis_selector(
        {'type': 'custom_scalar', 'functions': _norm_basis},
        data.d, output='vector'
    )
    S.infer_force_linear(
        basis_linear=force_b_, basis_linear_grad_v=force_grad_b_v_,
        M_mode='symmetric', G_mode='shift', diffusion_method='noisy',
        basis_names=names_
    )
    S.compute_force_error()

    # Ground truth (with force_boost to match simulated trajectories)
    Fy_gt, x_m, y_m = load_ground_truth(
        sim_entry['acoustic_field_file'],
        force_boost=sim_entry.get('force_boost', 1.0)
    )
    XX, YY = np.meshgrid(x_m, y_m, indexing='ij')
    Xeval  = np.stack([XX.ravel(), YY.ravel()], axis=-1)
    Veval  = np.zeros_like(Xeval)
    c_     = jnp.array(S.force_coefficients_full)

    def ef_(x, v):
        return jnp.einsum('a,iam->im', c_, force_b_(x[None], v[None]))[0]

    F_inf = np.array(vmap(ef_)(
        jnp.array(Xeval, dtype=jnp.float32),
        jnp.array(Veval, dtype=jnp.float32)
    ))
    Fy_inf = F_inf[:, 1].reshape(XX.shape)

    dx_m      = x_m[1] - x_m[0]
    sigma_2mm = 2e-3 / dx_m
    gt_sm     = gaussian_filter(Fy_gt,  sigma=sigma_2mm)
    inf_sm    = gaussian_filter(Fy_inf, sigma=sigma_2mm)

    r_full = np.corrcoef(Fy_gt.ravel(), Fy_inf.ravel())[0, 1]
    r_2mm  = np.corrcoef(gt_sm.ravel(), inf_sm.ravel())[0, 1]
    rmse   = np.sqrt(np.mean((Fy_inf - Fy_gt)**2))
    return r_full, r_2mm, rmse, S


# Run on all simulations (may take a few minutes)
results = []
for entry in traj_meta:
    print(f"Sim {entry['sim_index']:04d}  {entry['geometry']:5s}  {entry['n_transducers']}tx  ",
          end='', flush=True)
    r_full, r_2mm, rmse, _ = run_uli(entry)
    results.append({
        'sim_index':    entry['sim_index'],
        'geometry':     entry['geometry'],
        'n_transducers': entry['n_transducers'],
        'pearson_r':    float(r_full),
        'pearson_r_2mm': float(r_2mm),
        'rmse':         float(rmse),
    })
    print(f"r={r_full:.3f}  r_2mm={r_2mm:.3f}  RMSE={rmse:.3e}")


In [11]:
# Plot summary: Pearson r (2 mm smoothed) vs n_transducers, grouped by geometry
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
fig.suptitle('ULI Recovery: Pearson r at 2 mm smoothing vs. Number of Transducers',
             fontsize=13, fontweight='bold')

for ax, geo in zip(axes, ['well', 'slide']):
    geo_res = [r for r in results if r['geometry'] == geo]
    ntx_vals  = sorted(set(r['n_transducers'] for r in geo_res))
    for ntx in ntx_vals:
        rvals = [r['pearson_r_2mm'] for r in geo_res if r['n_transducers'] == ntx]
        ax.scatter([ntx] * len(rvals), rvals, s=60, zorder=3)
        ax.plot([ntx] * len(rvals), rvals, 'o', alpha=0.6)
    means = [np.mean([r['pearson_r_2mm'] for r in geo_res if r['n_transducers'] == ntx])
             for ntx in ntx_vals]
    ax.plot(ntx_vals, means, 'k--', lw=2, label='mean')
    ax.axhline(0, color='gray', lw=0.8, ls=':')
    ax.set_xlabel('Number of transducers')
    ax.set_ylabel('Pearson r (2 mm smooth)')
    ax.set_title(geo.capitalize())
    ax.legend()

plt.tight_layout()
plt.savefig('trajectoryData/uli_summary.png', dpi=150, bbox_inches='tight')
plt.show()

import json as _json
_json.dump(results, open('trajectoryData/uli_results.json', 'w'), indent=2)
print('Results saved to trajectoryData/uli_results.json')


## 7. Summary

This notebook demonstrated the full Bespoke Ultrasound inference pipeline:

| Step | Tool | Output |
|------|------|--------|
| Acoustic simulation | `kwaveTrainingDataGenerator.py` | RMS pressure, radiation force density (HDF5) |
| Trajectory simulation | `simulateTrajectories.py` | Underdamped Langevin cell paths (CSV) |
| ULI inference | this notebook | Recovered force field, Pearson r vs k-Wave |

### Current limitations & how to improve

The ULI inference operates on trajectories that only weakly explore the spatial force landscape:
- 5 cells each drift ~10 µm in 50 s, within a domain ~22 mm wide
- The polynomial order-1 basis can only recover a linear trend, not the standing-wave nodal pattern (~0.77 mm λ/2 period)
- Increasing `N_CELLS` (100–1000) distributed across the domain, or using higher-order polynomial/Fourier bases, would dramatically improve spatial resolution

**Parameter tuning for better ULI accuracy:**

| Parameter | Current | Suggested improvement |
|-----------|---------|----------------------|
| `N_CELLS` | 5 | 50–200 |
| `N_STEPS` | 5,000 | 10,000+ |
| Basis order | 1 | 3 (captures standing-wave curvature) |
| `GAMMA` | 1.0 s⁻¹ | calibrate from MSD of real tracks |

**Next steps toward experimental data:**
- Replace synthetic trajectories with real MG-63 (or A375) microscopy tracking data
- Extend to overdamped inference (`SFI.OverdampedLangevinInference`) for comparison — biologically appropriate when inertia << friction
- Use parsimonious model selection (Gerardos & Ronceray, arXiv:2501.10339) to find minimal basis without overfitting
- Validate against MSD analysis and phalloidin/vinculin IF cytoskeletal reorganization